In this notebook, I try to match the gutenberg corpus to the goodreads user data, trying to figure out usable data.


In [10]:
import pandas as pd
import requests
import time
from io import StringIO

GUTENBERG_CSV_URL = "https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv"

for attempt in range(5):
    try:
        r = requests.get(GUTENBERG_CSV_URL, timeout=60)
        r.raise_for_status()
        gutenberg_catalog = pd.read_csv(StringIO(r.text))
        break
    except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as e:
        print(f"Attempt {attempt+1} failed: {e}. Retrying...")
        time.sleep(5)

print(f"Gutenberg catalog: {len(gutenberg_catalog)} entries")
gutenberg_catalog.to_csv("gutenberg_list.csv", index=False)
print("Saved to gutenberg_list.csv")


Gutenberg catalog: 78842 entries
Saved to gutenberg_list.csv


In [11]:
import os
import pandas as pd

local_path = "goodreads_list.csv"

if os.path.exists(local_path):
    goodreads_books = pd.read_csv(local_path)
    print(f"Loaded existing {local_path}: {len(goodreads_books)} rows")
else:
    import requests, gzip, json, time
    GOODREADS_BOOKS_URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz"
    gz_path = "goodreads_books.json.gz"
    headers = {"User-Agent": "Mozilla/5.0"}
    for attempt in range(5):
        try:
            r = requests.get(GOODREADS_BOOKS_URL, headers=headers, stream=True, timeout=60)
            r.raise_for_status()
            with open(gz_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=16*1024*1024):
                    f.write(chunk)
            print("Download complete.")
            break
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as e:
            print(f"Attempt {attempt+1} failed: {e}. Retrying...")
            time.sleep(5)
    records = []
    with gzip.open(gz_path) as f:
        for i, line in enumerate(f):
            book = json.loads(line)
            records.append({
                "book_id": book.get("book_id"), "title": book.get("title"),
                "authors": book.get("authors"), "average_rating": book.get("average_rating"),
                "language_code": book.get("language_code"),
            })
            if i % 200000 == 0:
                print(f"Processed {i} books...")
    goodreads_books = pd.DataFrame(records)
    goodreads_books.to_csv(local_path, index=False)
    print(f"Saved {len(goodreads_books)} rows to {local_path}")


Loaded existing goodreads_list.csv: 2360655 rows


In [12]:
import pandas as pd
import re

gutenberg = pd.read_csv("gutenberg_list.csv")
goodreads = pd.read_csv("goodreads_list.csv")

# filter to English only, before doing anything else
gutenberg = gutenberg[gutenberg["Language"] == "en"]
goodreads = goodreads[goodreads["language_code"].isin(["eng", "en-US", "en-GB", "en-CA"])]

print(f"Gutenberg English books: {len(gutenberg)}")
print(f"Goodreads English books: {len(goodreads)}")

def normalize_title(title):
    if pd.isna(title):
        return ""
    title = title.lower()
    title = re.sub(r"[^a-z0-9\s]", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

gutenberg["title_norm"] = gutenberg["Title"].apply(normalize_title)
goodreads["title_norm"] = goodreads["title"].apply(normalize_title)

# drop blanks and very short (1-2 char) normalized titles - too generic/unreliable
gutenberg = gutenberg[gutenberg["title_norm"].str.len() > 2]
goodreads = goodreads[goodreads["title_norm"].str.len() > 2]

print(f"Gutenberg after cleaning: {len(gutenberg)}")
print(f"Goodreads after cleaning: {len(goodreads)}")

matched = gutenberg.merge(goodreads, on="title_norm", suffixes=("_gutenberg", "_goodreads"))
print(f"\nMatched (title only, English-filtered): {len(matched)}")

dup_check = matched["title_norm"].value_counts()
print("\nTop 10 most duplicated normalized titles:")
print(dup_check.head(10))

matched.to_csv("title_matched_books.csv", index=False)
print("Saved to title_matched_books.csv")

Gutenberg English books: 62478
Goodreads English books: 865919
Gutenberg after cleaning: 62463
Goodreads after cleaning: 865041

Matched (title only, English-filtered): 38721

Top 10 most duplicated normalized titles:
title_norm
poems                              2001
pride and prejudice                1170
a christmas carol                   655
dracula                             552
romeo and juliet                    546
the picture of dorian gray          544
selected poems                      472
alices adventures in wonderland     428
the wind in the willows             355
heart of darkness                   312
Name: count, dtype: int64
Saved to title_matched_books.csv


In [13]:
import pandas as pd

matched = pd.read_csv("title_matched_books.csv")

def primary_author(author_str):
    if pd.isna(author_str):
        return ""
    first = author_str.split(";")[0]
    first = first.split(",")[0] + "," + first.split(",")[1] if "," in first else first
    return first.strip().lower()

matched["author_key"] = matched["Authors"].apply(primary_author)
matched["book_key"] = matched["title_norm"] + "||" + matched["author_key"]

# FIX: one Goodreads book_id can title-match multiple Gutenberg rows (e.g. original + translation).
# Keep only ONE book_key per book_id so downstream merges can't fan out.
goodreads_to_bookkey = (
    matched[["book_id", "book_key"]]
    .drop_duplicates()
    .sort_values("book_key")
    .drop_duplicates(subset="book_id", keep="first")
)

# hard check: every book_id must map to exactly one book_key now
assert goodreads_to_bookkey["book_id"].is_unique, "book_id still not unique - fan-out still possible"

print(f"Total Goodreads editions mapped: {len(goodreads_to_bookkey)}")
print(f"Mapping to unique book_keys: {goodreads_to_bookkey['book_key'].nunique()}")

goodreads_to_bookkey.to_csv("goodreads_edition_to_bookkey.csv", index=False)
print("Saved to goodreads_edition_to_bookkey.csv")

Total Goodreads editions mapped: 22073
Mapping to unique book_keys: 5066
Saved to goodreads_edition_to_bookkey.csv


In [14]:
import os

local_path = "goodreads_interactions.csv"

if os.path.exists(local_path):
    print(f"Found existing {local_path}, skipping download.")
else:
    import requests, time
    INTERACTIONS_URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv"
    headers = {"User-Agent": "Mozilla/5.0"}
    for attempt in range(5):
        try:
            response = requests.get(INTERACTIONS_URL, headers=headers, stream=True, timeout=60)
            response.raise_for_status()
            total_size = int(response.headers.get("content-length", 0))
            print(f"Total file size: {total_size / 1e9:.2f} GB")
            downloaded = 0
            chunk_size = 16 * 1024 * 1024
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    f.write(chunk)
                    downloaded += len(chunk)
                    if downloaded % (200*1024*1024) < chunk_size:
                        print(f"Downloaded {downloaded / 1e9:.2f} GB / {total_size / 1e9:.2f} GB")
            print("Download complete.")
            break
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as e:
            print(f"Attempt {attempt+1} failed: {e}. Retrying from scratch...")
            time.sleep(5)


Found existing goodreads_interactions.csv, skipping download.


In [15]:
import pandas as pd

edition_map = pd.read_csv("goodreads_edition_to_bookkey.csv")
valid_book_ids = set(edition_map["book_id"].astype(str))
print(f"Filtering for {len(valid_book_ids)} known Goodreads book editions")

chunks_kept = []
chunk_size = 500_000

reader = pd.read_csv("goodreads_interactions.csv", chunksize=chunk_size, dtype={"user_id": str, "book_id": str})

for i, chunk in enumerate(reader):
    filtered = chunk[chunk["book_id"].isin(valid_book_ids)]
    if len(filtered) > 0:
        chunks_kept.append(filtered)
    if i % 20 == 0:
        print(f"Processed {i * chunk_size:,} rows so far, kept {sum(len(c) for c in chunks_kept):,}")

interactions = pd.concat(chunks_kept, ignore_index=True)
print(f"\nFinal filtered interactions: {len(interactions)}")
print(interactions.head(5))

interactions.to_csv("filtered_interactions.csv", index=False)
print("Saved to filtered_interactions.csv")

Filtering for 22073 known Goodreads book editions
Processed 0 rows so far, kept 7,479
Processed 10,000,000 rows so far, kept 135,481
Processed 20,000,000 rows so far, kept 260,553
Processed 30,000,000 rows so far, kept 389,248
Processed 40,000,000 rows so far, kept 516,580
Processed 50,000,000 rows so far, kept 642,406
Processed 60,000,000 rows so far, kept 768,131
Processed 70,000,000 rows so far, kept 894,228
Processed 80,000,000 rows so far, kept 1,018,629
Processed 90,000,000 rows so far, kept 1,143,145
Processed 100,000,000 rows so far, kept 1,266,366
Processed 110,000,000 rows so far, kept 1,389,302
Processed 120,000,000 rows so far, kept 1,512,354
Processed 130,000,000 rows so far, kept 1,635,640
Processed 140,000,000 rows so far, kept 1,757,763
Processed 150,000,000 rows so far, kept 1,882,406
Processed 160,000,000 rows so far, kept 2,005,117
Processed 170,000,000 rows so far, kept 2,128,565
Processed 180,000,000 rows so far, kept 2,255,327
Processed 190,000,000 rows so far, ke

In [16]:
import pandas as pd

interactions = pd.read_csv("filtered_interactions.csv", dtype={"user_id": str, "book_id": str})
edition_map = pd.read_csv("goodreads_edition_to_bookkey.csv", dtype={"book_id": str})

rated = interactions[interactions["rating"] > 0].copy()

# join to get the real consolidated book_key for each rating
rated = rated.merge(edition_map, on="book_id", how="left")

print(f"Ratings before dropping unmapped: {len(rated)}")
rated = rated.dropna(subset=["book_key"])
print(f"Ratings with valid book_key: {len(rated)}")

# a user may have rated multiple editions of the same book - combine into one (average rating)
consolidated = (
    rated.groupby(["user_id", "book_key"])
    .agg(rating=("rating", "mean"), num_editions_rated=("book_id", "count"))
    .reset_index()
)

print(f"\nConsolidated (user, real book) ratings: {len(consolidated)}")

# recompute num_rated_books correctly, based on real books not editions
rating_counts = consolidated.groupby("user_id").size().rename("num_rated_books")
consolidated = consolidated.merge(rating_counts, on="user_id")

usable = consolidated[consolidated["num_rated_books"] > 8].copy()

print(f"Unique users with >8 DISTINCT rated books: {usable['user_id'].nunique()}")
print(usable.head(10))

usable.to_csv("usable_interactions_consolidated.csv", index=False)
print("Saved to usable_interactions_consolidated.csv")

Ratings before dropping unmapped: 1496887
Ratings with valid book_key: 1496887

Consolidated (user, real book) ratings: 1421837
Unique users with >8 DISTINCT rated books: 26643
    user_id                                         book_key  rating  \
45   100012             as you like it||shakespeare, william     4.0   
46   100012               mans best friend||smith, evelyn e.     3.0   
47   100012                              mexico||hale, susan     3.0   
48   100012           the castle of otranto||walpole, horace     3.0   
49   100012               the house of mirth||wharton, edith     4.0   
50   100012            the odyssey||homer, 751? bce-651? bce     4.0   
51   100012  the time machine||wells, h. g. (herbert george)     5.0   
52   100012                          the trial||kafka, franz     3.0   
53   100012         treasure island||stevenson, robert louis     4.0   
119  100039           a tale of two cities||dickens, charles     3.0   

     num_editions_rated  num_r

In [17]:
import pandas as pd
import re

# STEP 1: fiction filter
df = pd.read_csv("gutenberg_list.csv")
fiction_prefixes = ("PN","PR","PS","PZ","PQ","PT","PG","PA","PL","PH")

def is_fiction(locc):
    if pd.isna(locc): return False
    codes = [c.strip() for c in re.split(r"[;,]", str(locc))]
    return any(c.startswith(fiction_prefixes) for c in codes)

df["is_fiction"] = df["LoCC"].apply(is_fiction)
fiction = df[df["is_fiction"]].copy()

# STEP 2-4: periodicals, poetry, index/bibliography filters
months = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
periodical_mask = (
    fiction["Title"].str.contains(r"\bMagazine\b|\bGazette\b|\bQuarterly\b", case=False, na=False, regex=True)
    | fiction["Title"].str.contains(months + r".{0,15}\bNo\.\s*\d", case=False, na=False, regex=True)
    | fiction["Title"].str.contains(r"\bVolume\s+\d+,?\s*Number\s+\d+", case=False, na=False, regex=True)
    | fiction["Subjects"].str.contains("Periodicals", case=False, na=False)
)
poetry_mask = fiction["Subjects"].str.contains(r"Poetry|Poems?\b|Verses?\b|Ballads?\b", case=False, na=False, regex=True)
index_mask = fiction["Subjects"].str.contains(r"\bindex|\bbibliograph|\bcatalogue\b", case=False, na=False, regex=True)

fiction["is_periodical"] = periodical_mask
fiction["is_poetry"] = poetry_mask
fiction["is_index"] = index_mask
clean = fiction[~periodical_mask & ~poetry_mask & ~index_mask].copy()
print(f"Raw: {len(df)} -> fiction: {len(fiction)} -> clean: {len(clean)}")

# STEP 5: English only
clean_en = clean[clean["Language"] == "en"].copy()
print(f"English-only: {len(clean_en)}")
clean_en.to_csv("gutenberg_fiction_final.csv", index=False)

# STEP 6: build book_key, match against ratings
ratings = pd.read_csv("usable_interactions_consolidated.csv")

def clean_author(a):
    if pd.isna(a): return ""
    first = str(a).split(";")[0]
    first = re.sub(r",?\s*\d{1,4}\??\s*(BCE|CE)?\s*-\s*\d{0,4}\??\s*(BCE|CE)?", "", first, flags=re.IGNORECASE)
    first = re.sub(r"\[.*?\]", "", first)
    first = re.sub(r",?\s*\b(graf|grafin|baron|baroness|comte|comtesse|conte|contessa|principe|principessa|sir|dame|lord|lady)\b\.?", "", first, flags=re.IGNORECASE)
    first = re.sub(r"\s{2,}", " ", first)
    return first.strip().strip(",").strip().lower()

def clean_title(t):
    if pd.isna(t): return ""
    t = str(t).split("\r")[0].split("\n")[0]
    t = re.sub(r"[^a-z0-9 ]", "", t.lower())
    return t.strip()

clean_en["title_key"] = clean_en["Title"].apply(clean_title)
clean_en["author_key"] = clean_en["Authors"].apply(clean_author)
clean_en["book_key"] = clean_en["title_key"] + "||" + clean_en["author_key"]

ratings["book_key_clean"] = ratings["book_key"].apply(
    lambda k: clean_title(k.split("||")[0]) + "||" + clean_author(k.split("||")[1]) if "||" in k else k
)

# FIX: dedupe on FULL book_key (title+author), and verify true uniqueness before merge
gut_dedup = clean_en.sort_values("Text#").drop_duplicates(subset="book_key", keep="first")
assert gut_dedup["book_key"].is_unique, "book_key still not unique after dedup - investigate"

matched = ratings.merge(
    gut_dedup[["Text#", "Title", "Authors", "book_key"]],
    left_on="book_key_clean", right_on="book_key", how="inner"
)

# FIX: hard check - merge must not multiply rows
assert len(matched) <= len(ratings), f"merge blew up rows: {len(ratings)} -> {len(matched)}"
if matched["book_key_clean"].duplicated().any():
    dup_keys = matched.loc[matched.duplicated(subset=["user_id","book_key_clean"], keep=False), "book_key_clean"].unique()
    print(f"WARNING: {len(dup_keys)} book_key_clean values still map to multiple Gutenberg rows - check these:")
    print(dup_keys[:10])

print(f"Original interactions: {len(ratings)} -> Matched: {len(matched)}")
print(f"Original unique books: {ratings['book_key'].nunique()} -> Matched unique books: {matched['book_key_clean'].nunique()}")
print(f"Original unique users: {ratings['user_id'].nunique()} -> Matched unique users: {matched['user_id'].nunique()}")

matched.to_csv("matched_interactions_final.csv", index=False)
print("Saved: gutenberg_fiction_final.csv, matched_interactions_final.csv")


Raw: 78842 -> fiction: 44492 -> clean: 39768
English-only: 29868
<StringArray>
[]
Length: 0, dtype: str
Original interactions: 332358 -> Matched: 273385
Original unique books: 1689 -> Matched unique books: 1284
Original unique users: 26643 -> Matched unique users: 26643
Saved: gutenberg_fiction_final.csv, matched_interactions_final.csv


In [18]:
import pandas as pd

m = pd.read_csv("matched_interactions_final.csv")

agg = m.groupby("user_id").agg(
    num_books_rated=("Text#", "size"),
    gutenberg_ids=("Text#", lambda x: ";".join(map(str, x))),
    goodreads_book_keys=("book_key_clean", lambda x: ";".join(x)),
    ratings=("rating", lambda x: ";".join(map(str, x)))
).reset_index()

# keep only users with 8 or more matched books rated
agg = agg[agg["num_books_rated"] >= 8].copy()

print(agg.shape)
print(agg.head(3).to_string())

agg.to_csv("user_book_ratings_summary_8plus.csv", index=False)
print("Saved to user_book_ratings_summary_8plus.csv")

(21631, 5)
   user_id  num_books_rated                                                                                                  gutenberg_ids                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           goodreads_book_keys                                                     